# Day 1 — From Model to Agent

## Daily project: Smart Research Assistant

This is the classroom master notebook for Day 1. Work from top to bottom: each section introduces one limitation, adds one system layer, and carries that improvement into the daily project.

### How to use this notebook

- Run the environment check and setup cells before beginning.
- Complete sections in order during class; optional provider comparisons are clearly marked.
- If Colab restarts, rerun the current section's import/setup cell before continuing.
- At each checkpoint, explain the observable change before moving forward.
- Use mock or cached mode first. Use the instructor-issued OpenRouter credit only for bounded live observations.

### Day 1 contents

1. [Your First Model Call](#day-1-section-1)
2. [Configuring Model Behaviour](#day-1-section-2)
3. [Structured Outputs](#day-1-section-3)
4. [Tool Calling](#day-1-section-4)
5. [Build the Agent Loop Manually](#day-1-section-5)
6. [Represent the Agent Loop with LangGraph](#day-1-section-6)
7. [Day 1 Project — Smart Research Assistant](#day-1-section-7)
8. [Complete the Manual Agent Loop](#day-1-section-8)

---


<a id="day-1-section-1"></a>

## 1.1 — Your First Model Call

We begin with the smallest useful AI application:

```text
Question → Model → Response
```

This is **not yet an agent**. No tool or action loop exists.

By the end, you can call the classroom model, identify the application's role, and understand the optional local/direct-provider alternatives.


## Before you begin

### Learning outcomes

Send one prompt through the classroom route and identify request, response, provider, model, and usage fields.

Architecture reference: [D01](../../diagrams/source/day_01.md).

### Expected observation

Mock output is deterministic; live wording varies, but a non-empty response and usage record should appear.


## Concept briefing

## Why this day exists

A language model is a generator, not an application. It receives a finite context and
predicts a continuation. It does not automatically know your files, execute Python, or
continue working until a goal is complete. An agentic application is created when host
code gives the model a limited set of possible actions, carries state between turns,
executes approved actions, and decides when the run must stop.

Day 1 removes the apparent magic from this process. By the end, students should be able
to point to the exact line that sends a request, the exact data that describes a tool,
the exact function that executes it, and the exact condition that terminates the loop.

## What a model call actually contains

A typical request contains a model identifier, ordered messages, optional tool schemas,
and generation controls. Messages are not merely a chat transcript. Their roles tell the
provider how each piece should be interpreted:

- `system`: standing instructions and boundaries;
- `user`: the current task or supplied information;
- `assistant`: previous model output, including tool requests;
- `tool`: an observation produced by host-executed code.

The provider serializes this request into a form the model can process. The model sees
tokens representing instructions, messages and tool descriptions. It does not receive a
live Python function. When it appears to "call" a tool, it is generating structured
tokens that name a function and propose arguments. The host application parses those
tokens, validates the arguments, applies policy, calls ordinary code, and returns the
result in another message.

This distinction is load-bearing:

```text
model proposes structured tokens
-> application validates and authorizes
-> Python executes
-> application records the observation
-> model sees the observation on the next call
```

If the model invents a tool name, supplies the wrong type, or requests a prohibited
action, nothing should happen unless the application accepts the request.


## The four course routes

1. **OpenRouter + GPT-OSS:** primary classroom route using your individually issued key.
2. **Ollama:** optional local-provider comparison for suitable computers.
3. **Direct OpenAI API:** optional for students with their own API access.
4. **Mock mode:** deterministic testing without network calls or cost.

The remaining guided notebooks use OpenRouter consistently. Never paste a key into a notebook or commit `.env`.

## Part A — OpenRouter (classroom default)

Before class, place your issued key in the repository's `.env` file:

```dotenv
OPENROUTER_API_KEY=your_individual_course_key
OPENROUTER_MODEL=openai/gpt-oss-120b
```

Your key has a course-wide lifetime spending limit. Do not share it.

In [ ]:
# Run once if required, then restart the kernel.
# %pip install -q openai python-dotenv
import os
from types import SimpleNamespace
from dotenv import load_dotenv
from openai import OpenAI
load_dotenv()
COURSE_MODEL=os.getenv("OPENROUTER_MODEL","openai/gpt-oss-120b")
api_key=os.getenv("OPENROUTER_API_KEY")
client=OpenAI(base_url="https://openrouter.ai/api/v1",api_key=api_key) if api_key else None
print("Route:","OpenRouter" if client else "mock fallback")


In [ ]:
question="Explain recursion in two sentences for a beginner."
if client:
    response=client.chat.completions.create(model=COURSE_MODEL,messages=[{"role":"user","content":question}],
        max_tokens=300,extra_body={"reasoning":{"effort":"low","exclude":True},"provider":{"sort":"price"}})
    answer=response.choices[0].message.content
else:
    response=None
    answer="Recursion is when a function solves a problem by calling itself on a smaller version. It needs a base case so the calls eventually stop."
print(answer)


### Observe

- Which object sends the request?
- Which value selects the model?
- Where is response length bounded?
- Did the model execute a Python function?
- Why is low reasoning sufficient for this simple request?

The application sends messages and controls the request. The model generates text.

In [ ]:
if response:
    print(response.usage)
else:
    print({"provider":"mock","prompt_tokens":0,"completion_tokens":0,"cost_usd":0.0})


## Part B — Ollama (optional provider-portability comparison)

If your computer can run a local model, install Ollama and its Python package, download the instructor-approved comparison model, and run the same prompt. This section is optional; the course does not assume every laptop can run it well.

In [ ]:
# Optional local route:
# %pip install -q ollama
# from ollama import chat
# local_response = chat(
#     model="qwen3:4b",  # replace with the instructor-approved comparison model
#     messages=[{"role": "user", "content": question}],
# )
# print(local_response.message.content)

## Part C — Direct OpenAI API (optional alternative)

Students with their own OpenAI API project can use the official Python SDK and Responses API. A ChatGPT subscription and API billing are separate. The SDK reads `OPENAI_API_KEY`; never write the key in this notebook.

Official guide: https://platform.openai.com/docs/quickstart

In [ ]:
# Optional direct OpenAI route:
# from openai import OpenAI
# direct_client = OpenAI()  # reads OPENAI_API_KEY
# direct_response = direct_client.responses.create(
#     model=os.getenv("OPENAI_MODEL", "gpt-5.6-luna"),
#     input=question,
# )
# print(direct_response.output_text)

## Part D — Mock mode

A mock is useful for testing Python flow during an outage, but it does not measure real model quality.

In [ ]:
def mock_model(prompt: str) -> str:
    return f"Prepared mock response for: {prompt}"

print(mock_model(question))

## Exercise and checkpoint

Ask for a two-sentence explanation, one example, and one limitation from your engineering discipline. Run it twice and note what changes.

We built `application → model → text response`. The limitation is that free-form text is not a dependable application data structure.

## Required live observation

Send one bounded prompt through the issued OpenRouter route and save the response plus usage record. If service access fails, inspect the instructor-captured trace and continue in mock mode.


## Your turn

Change one prompt constraint and compare outputs without changing providers.

## Recap

A model call generates output from supplied context; it does not create an agent.


---

### Section 1.1 checkpoint

Before continuing, confirm that you can state: **what limitation we observed, what layer we added, and what evidence shows the improvement worked.**


<a id="day-1-section-2"></a>

## 1.2 — Configuring Model Behaviour

Previously, we sent one question and received unrestricted text. Now we use message roles and clear constraints.

```text
System instructions + user request → model → better-shaped text
```

The guided notebooks use the issued OpenRouter key. Optional Ollama and direct OpenAI setup remains in Notebook 01.


## Before you begin

### Learning outcomes

Separate standing instructions from the current task and observe temperature/output constraints.

Architecture reference: [D01](../../diagrams/source/day_01.md).

### Expected observation

The constrained response follows the requested format more reliably than the broad prompt.


## Learning objectives

Distinguish system and user messages, configure answer organization and length, and observe that instructions improve consistency without guaranteeing a schema.

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
load_dotenv()
api_key=os.getenv("OPENROUTER_API_KEY")
client=OpenAI(base_url="https://openrouter.ai/api/v1",api_key=api_key) if api_key else None
COURSE_MODEL=os.getenv("OPENROUTER_MODEL","openai/gpt-oss-120b")
def ask(messages,max_tokens=400):
    if not client:
        constrained=any(message.get("role")=="system" for message in messages)
        return "Definition: An agent chooses bounded actions.\nExample: It requests a calculator tool.\nLimitation: Host code must control execution." if constrained else "An AI agent uses a model and tools to work toward a goal."
    return client.chat.completions.create(model=COURSE_MODEL,messages=messages,max_tokens=max_tokens,
        extra_body={"reasoning":{"effort":"low","exclude":True}}).choices[0].message.content
print("Route:","OpenRouter" if client else "mock fallback")


## Build: begin with a broad request and observe its variability

In [ ]:
print(ask([{"role": "user", "content": "Explain an AI agent."}]))

## Improve: separate standing instructions from the current task

A system message describes how the model should behave for the call. A user message contains the current request.

In [ ]:
messages = [
    {"role": "system", "content": (
        "You teach engineering students new to agentic AI. Use plain language. "
        "Give exactly three short sections: Definition, Example, and Limitation."
    )},
    {"role": "user", "content": "Explain an AI agent."},
]
print(ask(messages))

### Observe

Did all sections appear? Is it beginner-friendly? Does rerunning produce identical punctuation? Clear instructions help, but application boundaries still require validation.

## Break it

Ask for a Python dictionary with exact keys. Can the application safely assume every run returns parsable Python or JSON without extra text?

In [ ]:
print(ask([{"role": "user", "content": (
    "Explain an AI agent as a dictionary with exactly the keys definition, example, and limitation."
)}]))

## Exercise and checkpoint

Create a system message for your engineering discipline requiring a beginner explanation, example, limitation, and at most 150 words. Test two questions.

Instructions shape output but are not a software contract. Next we add schema-constrained output and validation.

## Your turn

Change one instruction at a time and record which behavior changes.

## Recap

Configuration shapes generation but does not guarantee truth or safety.


---

### Section 1.2 checkpoint

Before continuing, confirm that you can state: **what limitation we observed, what layer we added, and what evidence shows the improvement worked.**


<a id="day-1-section-3"></a>

## 1.3 — Structured Outputs

Clear instructions improved answers but did not create a reliable application contract. Now we build:

```text
Question → model → schema-shaped JSON → Pydantic validation → Python object
```


## Before you begin

### Learning outcomes

Define a Pydantic contract, request structured data, and handle validation failure.

Architecture reference: [D02](../../diagrams/source/day_01.md).

### Expected observation

Valid data becomes a typed object; plausible data outside field constraints is rejected.


## Concept briefing

## Why structured output matters

Free-form text is useful for people but unreliable for software. A program cannot safely
assume every response contains the same headings, fields or value types. A schema turns
this ambiguity into a contract. Validation does not make the model correct; it makes a
particular class of failure visible.

Consider a confidence field. The sentence "confidence is high" may be understandable to
a person but difficult to compare. A schema can require a number between 0 and 1. If the
model returns `4.5`, validation rejects it instead of quietly sending bad data deeper into
the application.

The correct mental model is:

- schema validity asks whether the response has an acceptable shape;
- factual evaluation asks whether its claims are correct;
- policy asks whether a requested action is permitted.

These are different checks and should not be collapsed into one model prompt.


## Learning objectives

Explain why formatted text is not automatically valid data, define a Pydantic contract, request JSON Schema output through OpenRouter, and handle invalid data.

In [ ]:
import json,os
from types import SimpleNamespace
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel,Field,ValidationError
load_dotenv(); api_key=os.getenv("OPENROUTER_API_KEY")
client=OpenAI(base_url="https://openrouter.ai/api/v1",api_key=api_key) if api_key else None
COURSE_MODEL=os.getenv("OPENROUTER_MODEL","openai/gpt-oss-120b")
print("Route:","OpenRouter" if client else "mock fallback")


## Build: ask for JSON using words only

This often appears to work, but fields, types, and extra prose are not guaranteed.

In [ ]:
if client:
    plain=client.chat.completions.create(model=COURSE_MODEL,messages=[{"role":"user","content":"Explain an AI agent. Return JSON with topic, summary, key_points, and confidence."}],max_tokens=500,extra_body={"reasoning":{"effort":"low","exclude":True}})
    plain_text=plain.choices[0].message.content
else:
    plain_text='{"topic":"AI agents","summary":"A model-guided application","key_points":["May request tools"],"confidence":0.8}'
print(plain_text)


## Break it mentally

What if confidence is `high`, key points are one paragraph, Markdown surrounds the JSON, or a required field is missing? These are application-data failures.

## Improve: define the contract with Pydantic

In [ ]:
class ResearchSummary(BaseModel):
    topic: str
    summary: str
    key_points: list[str] = Field(min_length=1, max_length=5)
    confidence: float = Field(ge=0.0, le=1.0)

schema = ResearchSummary.model_json_schema()
schema

## Request schema-constrained output

OpenRouter standardizes structured-output requests for compatible models/providers. The application still validates the returned boundary.

In [ ]:
if client:
    response=client.chat.completions.create(model=COURSE_MODEL,messages=[{"role":"user","content":"Explain an AI agent for a beginner with two or three key points."}],response_format={"type":"json_schema","json_schema":{"name":"research_summary","strict":True,"schema":schema}},max_tokens=600,extra_body={"reasoning":{"effort":"low","exclude":True},"provider":{"require_parameters":True}})
    response_text=response.choices[0].message.content
else:
    response_text=json.dumps({"topic":"AI agents","summary":"An application that uses a model to choose bounded actions.","key_points":["The host executes tools","The loop needs limits"],"confidence":0.9})
result=ResearchSummary.model_validate_json(response_text)
result


In [ ]:
print(result.topic)
print(result.confidence)
for number, point in enumerate(result.key_points, start=1):
    print(f"{number}. {point}")

## Observe validation rejecting plausible but invalid data

In [ ]:
invalid_data = '''{
  "topic": "AI agents",
  "summary": "A short summary",
  "key_points": ["Uses a model"],
  "confidence": 4.5
}'''

try:
    ResearchSummary.model_validate_json(invalid_data)
except ValidationError as error:
    print(error)

## Exercise and checkpoint

Create `EngineeringConcept` with name, plain explanation, one-to-three applications, and difficulty from 1–5. Request and validate one concept. Invalid difficulty and empty applications must fail.

We now have `model output → schema validation → Python object`. The model still cannot obtain outside information or reliably perform calculations; tools solve that next.

## Your turn

Add one constrained field and deliberately supply an invalid value.

## Recap

A schema makes failure visible; it does not make model claims correct.


---

### Section 1.3 checkpoint

Before continuing, confirm that you can state: **what limitation we observed, what layer we added, and what evidence shows the improvement worked.**


<a id="day-1-section-4"></a>

## 1.4 — Tool Calling

A model can generate text, but it cannot directly execute your Python functions. We will let it **request** one safe calculator tool.

```text
User → model requests tool → Python validates and executes → result returns to model
```

By the end, you can define a tool schema, inspect a model request, execute it in Python, and return the observation.


## Before you begin

### Learning outcomes

Distinguish a model tool request from host validation and Python execution.

Architecture reference: [D03](../../diagrams/source/day_01.md).

### Expected observation

The model returns a name and arguments; the calculator runs only after validation in application code.


In [ ]:
import ast,json,operator,os
from types import SimpleNamespace
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel,Field,ValidationError
load_dotenv(); api_key=os.getenv("OPENROUTER_API_KEY")
client=OpenAI(base_url="https://openrouter.ai/api/v1",api_key=api_key) if api_key else None
COURSE_MODEL=os.getenv("OPENROUTER_MODEL","openai/gpt-oss-120b")
print("Route:","OpenRouter" if client else "mock fallback")


## Build a safe calculator

Never use unrestricted `eval()` on model-generated input. This deliberately small evaluator accepts numbers and basic arithmetic operators only.

In [ ]:
BINARY = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv}
UNARY = {ast.UAdd: operator.pos, ast.USub: operator.neg}

def evaluate_node(node):
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return float(node.value)
    if isinstance(node, ast.BinOp) and type(node.op) in BINARY:
        return BINARY[type(node.op)](evaluate_node(node.left), evaluate_node(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in UNARY:
        return UNARY[type(node.op)](evaluate_node(node.operand))
    raise ValueError("Only basic arithmetic is allowed")

def calculator(expression: str) -> str:
    value = evaluate_node(ast.parse(expression, mode="eval").body)
    return str(int(value)) if value.is_integer() else str(value)

calculator("12 * 7")

## Describe the tool to the model with a schema

In [ ]:
class CalculatorArguments(BaseModel):
    expression: str = Field(min_length=1, max_length=100)

calculator_tool = {
    "type": "function",
    "function": {
        "name": "calculator",
        "description": "Evaluate basic arithmetic instead of calculating mentally.",
        "parameters": {
            "type": "object",
            "properties": {"expression": {"type": "string"}},
            "required": ["expression"],
            "additionalProperties": False,
        },
    },
}

## Ask the model

The returned `tool_calls` value is a request—not evidence that anything has executed.

In [ ]:
messages=[{"role":"user","content":"What is 12 * 7? Use the calculator."}]
if client:
    first=client.chat.completions.create(model=COURSE_MODEL,messages=messages,tools=[calculator_tool],max_tokens=400,
        extra_body={"reasoning":{"effort":"low","exclude":False},"provider":{"require_parameters":True}})
    assistant_message=first.choices[0].message
else:
    call=SimpleNamespace(id="mock-calculator",function=SimpleNamespace(name="calculator",arguments='{"expression":"12 * 7"}'))
    assistant_message=SimpleNamespace(tool_calls=[call],model_dump=lambda **kwargs:{"role":"assistant","content":"","tool_calls":[{"id":call.id,"type":"function","function":{"name":"calculator","arguments":call.function.arguments}}]})
assistant_message.tool_calls


## Validate and execute in Python

In [ ]:
call = assistant_message.tool_calls[0]
arguments = CalculatorArguments.model_validate_json(call.function.arguments)
tool_output = calculator(arguments.expression)
print(call.function.name, arguments.expression, tool_output)

## Return the observation to the model

Append the assistant's original tool request and a tool-role result so the model receives the complete sequence.

In [ ]:
messages.append(assistant_message.model_dump(exclude_none=True))
messages.append({"role":"tool","tool_call_id":call.id,"content":tool_output})
if client:
    final=client.chat.completions.create(model=COURSE_MODEL,messages=messages,tools=[calculator_tool],max_tokens=300,
        extra_body={"reasoning":{"effort":"low","exclude":True}})
    final_text=final.choices[0].message.content
else:
    final_text=f"The calculator result is {tool_output}."
print(final_text)


## Break and inspect

Try malformed arguments and `__import__('os').getcwd()`. The schema checks shape; the tool implementation enforces what operations are permitted. Both layers matter.

## Exercise and checkpoint

Add a `convert_celsius_to_fahrenheit` tool with one numeric argument. Inspect the request before executing it.

We now have one complete tool interaction. The limitation is that the code assumes exactly one request and one tool call. A manual agent loop generalizes it next.

## Your turn

Send an unsupported argument and prove the function is not executed.

## Recap

The model requests; the host validates, authorizes, and executes.


---

### Section 1.4 checkpoint

Before continuing, confirm that you can state: **what limitation we observed, what layer we added, and what evidence shows the improvement worked.**


<a id="day-1-section-5"></a>

## 1.5 — Build the Agent Loop Manually

One hardcoded tool interaction cannot handle an unknown number of steps. We now build the mechanism that makes this application agentic:

```text
Model → decide → tool → observation → model → ... → final answer
```

The loop is controlled by Python. It has a step limit and explicit failure status.


## Before you begin

### Learning outcomes

Trace repeated model/tool turns and prove the host step limit stops execution.

Architecture reference: [D04](../../diagrams/source/day_01.md).

### Expected observation

Every tool result is appended before the next model call; forced looping ends at max_steps.


## Concept briefing

## Why the application owns termination

After one tool result, the model may ask for another tool or return a final answer. That
creates a loop whose length is not known in advance. It is tempting to write "stop when
finished" in the system message and trust the model. That is not an execution limit. A
confused model can repeat the same request, alternate between tools, or continue refining
an already adequate answer. Each turn consumes time, tokens and money.

Host code therefore enforces a maximum number of steps. Reaching the limit is not the
same as crashing. A good runtime returns a visible status such as `max_steps` together
with the partial trace. Reporting incomplete work honestly is safer than pretending the
run completed.

## Error compounding

Multi-step systems amplify small error rates. Suppose, only for illustration, that each
model decision has a 95% chance of being acceptable and that errors are independent. The
chance that ten decisions are all acceptable is:

```text
0.95 ^ 10 = approximately 0.60
```

The independence assumption is simplistic, but the lesson is useful: a system with many
model decisions can be much less reliable than any single impressive response suggests.
This motivates bounded loops, deterministic validation, fewer calls, clear tools and
evaluation of complete trajectories rather than isolated answers.


## Learning objectives

Trace a multi-step agent run, explain reason/action/observation in application terms, and show why termination and validation belong outside the model.

In [ ]:
import os
import sys
from pathlib import Path

# Locate the Day 1 project whether the notebook starts from the repository root or notebooks folder.
here = Path.cwd().resolve()
candidates = [here, here / "day_01_model_tools_agent", here.parent]
project_root = next(path for path in candidates if (path / "src" / "research_agent").exists())
sys.path.insert(0, str(project_root / "src"))

from research_agent.agent import AgentRunner
from research_agent.providers import OpenRouterProvider
from research_agent.tools import default_tool_registry

## Inspect the reusable pieces

The tool registry contains ordinary Python functions plus descriptions and argument schemas. The provider makes model requests. The runner owns the loop.

In [ ]:
tools = default_tool_registry()
for name, tool in tools.items():
    print(name, "→", tool.definition.description)

## Run a question requiring two tools

The provider reads the issued OpenRouter key from `.env`/the environment. The runner stops after at most five model turns.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

runner = AgentRunner(
    provider=OpenRouterProvider(),
    tools=tools,
    max_steps=5,
)
result = runner.run("Explain an AI agent using the local notes and calculate 12 * 7.")
print(result.status, result.steps, result.error)

## Observe every message in the loop

In [ ]:
for index, message in enumerate(result.messages):
    requested = [call.name for call in message.tool_calls]
    print(f"{index:02d} role={message.role:9} tool_requests={requested}")
    if message.role == "tool":
        print("   observation:", message.content[:160])

In [ ]:
print(result.response.model_dump_json(indent=2) if result.response else result.error)
print("Usage:", result.usage.model_dump())

## Read the control flow

Open `src/research_agent/agent.py` and identify: model call, final-answer validation, tool lookup, argument validation, duplicate-call check, and maximum-step termination.

The model proposes the next action. The application decides whether and how it is executed.

## Break it safely

Run with `max_steps=1`. Then request both explanation and calculation. Observe the explicit `max_steps` status instead of allowing an unbounded loop.

In [ ]:
limited = AgentRunner(OpenRouterProvider(), tools, max_steps=1)
limited_result = limited.run("Explain an AI agent using notes and calculate 12 * 7.")
print(limited_result.status, limited_result.error)

## Exercise and behaviour checks

Test: one direct question, one calculator question, one notes question, and one two-tool question. Record expected tool, actual tools, final schema validity, status, and steps.

The manual loop is now understandable but increasingly difficult to visualize and extend. Next we express the same mechanism as a graph—without replacing provider calls with LangChain abstractions.

## Your turn

Set max_steps to one and explain the incomplete trace.

## Recap

An agent loop is bounded application control around model decisions.


---

### Section 1.5 checkpoint

Before continuing, confirm that you can state: **what limitation we observed, what layer we added, and what evidence shows the improvement worked.**


<a id="day-1-section-6"></a>

## 1.6 — Represent the Agent Loop with LangGraph

We already understand the loop. LangGraph now gives it an explicit state-and-transition structure:

```text
START → model ──final──→ END
          └─tool request→ tools → model
          └─step limit──→ limit → END
```

LangGraph is not the agent's intelligence and does not replace the provider API. It organizes execution.


## Before you begin

### Learning outcomes

Represent the same loop as state, nodes, edges, and conditional routing.

Architecture reference: [D05](../../diagrams/source/day_01.md).

### Expected observation

The final state contains the message history and a terminal route.


## Concept briefing

## Workflow or agent?

Not every problem needs an agent. Use ordinary code or a deterministic workflow when the
steps and decision rules are known. Use a hybrid workflow when most steps are fixed but
one bounded judgment benefits from a model. Consider an agent when the next action cannot
be fully predetermined, the action set is small, failures are containable, and success
can be evaluated.

Ask:

1. Are the steps known in advance?
2. Can normal code make the decision reliably?
3. Does the model genuinely add judgment rather than decoration?
4. What is the consequence of a wrong action?
5. Is there a strict step and tool boundary?
6. Can we observe and evaluate the result?

If these questions have weak answers, the correct design is often a workflow, not an
agent.


## Learning objectives

Explain node, edge, conditional edge, and state; map the manual loop to a graph; and confirm that graph execution preserves the same model/tool responsibilities.

In [ ]:
# Install once if needed:
# %pip install -q langgraph

import sys
from pathlib import Path
from dotenv import load_dotenv

here = Path.cwd().resolve()
candidates = [here, here / "day_01_model_tools_agent", here.parent]
project_root = next(path for path in candidates if (path / "src" / "research_agent").exists())
sys.path.insert(0, str(project_root / "src"))
load_dotenv()

from research_agent.agent import SYSTEM_MESSAGE
from research_agent.graph import build_graph
from research_agent.providers import OpenRouterProvider
from research_agent.schemas import Message
from research_agent.tools import default_tool_registry

## State is application-owned data

Our graph state carries messages, current step count, maximum steps, final validated response, and error. Nodes read state and return updates.

In [ ]:
initial_state = {
    "messages": [
        Message(role="system", content=SYSTEM_MESSAGE),
        Message(role="user", content="Explain an AI tool using notes and calculate 12 * 7."),
    ],
    "steps": 0,
    "max_steps": 5,
    "final_response": None,
    "error": None,
}
initial_state

## Compile the graph

The implementation uses `StateGraph`, Python node functions, and conditional edges. Provider calls remain plain calls through `OpenRouterProvider`; no LangChain agent, chain, LCEL, or memory abstraction is used.

In [ ]:
graph = build_graph(OpenRouterProvider(), default_tool_registry())
print(graph.get_graph().draw_mermaid())

## Invoke the graph and inspect final state

In [ ]:
final_state = graph.invoke(initial_state)
print("steps:", final_state["steps"])
print("error:", final_state["error"])
print(final_state["final_response"].model_dump_json(indent=2) if final_state["final_response"] else "No final response")

## Observe the route from messages

In [ ]:
for message in final_state["messages"]:
    requests = [call.name for call in message.tool_calls]
    print(message.role, requests, message.content[:100])

## Compare manual loop and graph

| Manual loop | Graph |
|---|---|
| `for` iteration | model node visited repeatedly |
| `if tool_calls` | conditional edge |
| execute functions | tools node |
| return result | edge to END |
| step counter | state field and limit route |

The architecture is the same; the representation is more explicit.

## Exercise and checkpoint

Set `max_steps` to 1 and inspect the limit result. Then read `src/research_agent/graph.py` and label its model node, tools node, routing function, and edges.

We use LangGraph because later projects need state, branching, interrupts, and checkpoints—not because it magically creates an agent.

## Your turn

Change one routing condition to a safe failure and inspect the final state.

## Recap

LangGraph represents orchestration; it does not replace tools, policy, or evaluation.


---

### Section 1.6 checkpoint

Before continuing, confirm that you can state: **what limitation we observed, what layer we added, and what evidence shows the improvement worked.**


<a id="day-1-section-7"></a>

## 1.7 — Day 1 Project — Smart Research Assistant

We now assemble the layers introduced today:

```text
OpenRouter model → structured tool request → validated Python tool
→ observation → bounded agent loop → validated final response
```

A working demo is not enough. We will run a small behaviour suite and record what the system actually did.


## Before you begin

### Learning outcomes

Run the bounded research project and explain model, tool, loop, validation, and termination boundaries.

Architecture reference: [D01–D05](../../diagrams/source/day_01.md).

### Expected observation

The behavior suite completes in mock mode and reports usage without spending API credit.


## Concept briefing

## What to carry into Day 2

Day 1 creates a bounded model-and-tool system, but the model still relies on information
inside its request or learned during training. Day 2 introduces external knowledge. The
agent loop remains the same; the new question is how to retrieve the right evidence and
prove the answer used it.


## Completion criteria

The assistant must accept a question, use zero or more supplied tools, return observations to the model, validate final output, stop within five turns, and expose status, steps, tools, token usage, and provider-reported cost.

In [ ]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

here = Path.cwd().resolve()
candidates = [here, here / "day_01_model_tools_agent", here.parent]
project_root = next(path for path in candidates if (path / "src" / "research_agent").exists())
sys.path.insert(0, str(project_root / "src"))
load_dotenv()

from research_agent.agent import AgentRunner
from research_agent.providers import MockModelProvider, OpenRouterProvider
from research_agent.tools import default_tool_registry

## Select real or mock execution

Use OpenRouter for model behaviour. Use mock mode while debugging application code or during an outage. Mock results must not be reported as model-quality evidence.

In [ ]:
USE_MOCK = False  # Change to True only for deterministic/offline testing.
provider = MockModelProvider() if USE_MOCK else OpenRouterProvider()
runner = AgentRunner(provider, default_tool_registry(), max_steps=5)

## Run the completed project

In [ ]:
project_result = runner.run("Explain what an AI agent is using local notes, then calculate 12 * 7.")
print(project_result.response.model_dump_json(indent=2) if project_result.response else project_result.error)
print("status:", project_result.status)
print("model turns:", project_result.steps)
print("usage:", project_result.usage.model_dump())

## Behaviour suite

These checks are intentionally small. Formal golden-set evaluation begins on Day 2.

In [ ]:
cases = [
    {"id": "direct", "question": "Give a brief greeting.", "expected_tools": []},
    {"id": "calculation", "question": "Calculate 12 * 7.", "expected_tools": ["calculator"]},
    {"id": "knowledge", "question": "Use local notes to explain an AI tool.", "expected_tools": ["search_local_notes"]},
    {"id": "two_tools", "question": "Use notes to explain an AI agent and calculate 12 * 7.", "expected_tools": ["calculator", "search_local_notes"]},
]

In [ ]:
records = []
for case in cases:
    result = runner.run(case["question"])
    actual_tools = result.response.tools_used if result.response else []
    records.append({
        "case": case["id"],
        "status": result.status,
        "schema_valid": result.response is not None,
        "expected_tools": case["expected_tools"],
        "actual_tools": actual_tools,
        "tool_check": set(actual_tools) == set(case["expected_tools"]),
        "steps": result.steps,
        "tokens": result.usage.prompt_tokens + result.usage.completion_tokens,
        "cost_usd": round(result.usage.cost_usd, 6),
    })

for record in records:
    print(record)

## Interpret failures

- Wrong tool: model-selection behaviour or unclear tool description.
- Invalid arguments: schema/model boundary failure.
- Tool error: Python execution failure.
- Invalid final response: output-contract failure.
- Maximum steps: termination/control failure.

Do not call every failure a 'hallucination.' Locate the failing layer.

## Optional provider-portability comparison

Students with suitable hardware may run the same cases through `OllamaProvider`. Compare tool selection, schema validity, model turns, and elapsed time. The architecture remains stable even when model capability changes.

## Final reflection

Explain in your own words:

1. Why is a model call not automatically an agent?
2. Who executes a tool?
3. Why validate tool arguments and final output?
4. Who decides when the loop must stop?
5. What did LangGraph change, and what did it not change?

Day 1 made a model **do** something. Day 2 gives the agent grounded engineering knowledge.

## Your turn

Add one behavior case and one tool failure case with objective assertions.

## Recap

The project is an application-specific agent, not yet a reusable harness.


---

### Section 1.7 checkpoint

Before continuing, confirm that you can state: **what limitation we observed, what layer we added, and what evidence shows the improvement worked.**


<a id="day-1-section-8"></a>

## 1.8 — Complete the Manual Agent Loop

This is an individual implementation lab. It uses no API key.


## Why this mechanism matters

A tool-using agent is an application-controlled loop. The model proposes either a tool request or a final answer; Python validates, dispatches, records the observation, and decides whether another step is allowed.

## Contract

Implement `run_agent`. Reject unknown tools, append every tool result to `messages`, return final text, and raise `RuntimeError` when `max_steps` is exhausted.

Before coding, write one sentence predicting the easiest failure to make.

In [ ]:
def run_agent(model, tools, messages, max_steps=4):
    """Run a bounded observe-dispatch-append loop and return final text."""
    # TODO: repeat for at most max_steps
    # TODO: ask model(messages) for the next response
    # TODO: return response["text"] for type == "final"
    # TODO: validate and dispatch requests of type == "tool"
    # TODO: append {"role": "tool", "name": ..., "content": ...}
    raise NotImplementedError("Complete the agent loop")

## Behavioural check

Run this only after completing the starter cell. A passing check proves the listed contract examples, not every possible input.

In [ ]:
calls = []
def add(a, b):
    calls.append((a, b)); return a + b

responses = iter([
    {"type": "tool", "name": "add", "arguments": {"a": 2, "b": 3}},
    {"type": "final", "text": "The result is 5."},
])
history = [{"role": "user", "content": "Add 2 and 3"}]
assert run_agent(lambda messages: next(responses), {"add": add}, history) == "The result is 5."
assert calls == [(2, 3)]
assert any(item.get("role") == "tool" for item in history)
print("PASS")

## Explain and extend

Why must the application, rather than the model, own tool execution and termination? Add a test for an unknown tool and one for a model that never returns a final answer.

---

### Section 1.8 checkpoint

Before continuing, confirm that you can state: **what limitation we observed, what layer we added, and what evidence shows the improvement worked.**


## Day 1 completion checklist

- [ ] I can explain how every section contributes to the **Smart Research Assistant**.
- [ ] I ran the deterministic path and at least one required live observation or classroom fallback.
- [ ] I completed the pivotal exercise without copying the reference implementation.
- [ ] I can identify the system state, safety boundary, and evidence used to judge the result.
